# Task 07 — verified v1 diagnosis and a frozen v2 comparison

This notebook is **unexecuted preparation**, not v2 performance evidence. Choose
GPU and run cells in order. It loads the SAME selected Task 03B epoch-2 classifier
from Drive, never retrains, changes selection, substitutes another model or reads
official test text. All new outputs go directly to a new durable Task 07 folder.

The original Task 06 result ZIP is independently verified and its tables recomputed.
RAW, TAT, REP2, NO_CONTEXT and NASN2 follow the already frozen Task 07 protocol.
NO_CONTEXT is the simple combined baseline/context ablation. Primary interpretation
requires paired superiority over every simple baseline and clean performance,
plus semantic/provenance evidence; failed performance gates recommend abandoning
the improvement claim or reframing rather than adding rules.


In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, subprocess, sys, uuid
CODE_REF='304e54bd4ba0f6fa9b012244de090ee66d84b014'
REPO=Path('/content/nasn-task07')
RUN_ID='nasn-v2-controlled-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
def call(command, **kwargs):
    result=subprocess.run([str(x) for x in command],text=True,capture_output=True,**kwargs)
    print(result.stdout, end=''); print(result.stderr, end='',file=sys.stderr)
    result.check_returncode()
    return result
if not REPO.exists():
    call(['git','clone','--no-checkout','https://github.com/haneenbiate/-.git',REPO])
    call(['git','-C',REPO,'checkout','--detach',CODE_REF])
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==CODE_REF, 'Existing checkout has another revision: choose a fresh runtime'
assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Pinned tracked files changed'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert os.path.ismount('/content/drive'), 'A real mounted Drive is required'
ROOT=Path('/content/drive/MyDrive/NASN/task07')/RUN_ID
ROOT.mkdir(parents=True,exist_ok=False)
probe=ROOT/'.storage_probe';probe.write_text('durable write/read probe');assert probe.read_text()=='durable write/read probe';probe.unlink()
BASELINE=REPO/'reports/task03b/marbert-v1-20261009T213143Z-be8b8811'
CHECKPOINT=Path(json.loads((BASELINE/'run_status.json').read_text())['best_checkpoint'])
# Change this path only if the same selected checkpoint was moved on Drive.
# Every model/tokenizer/config byte must match its original registry.
assert CHECKPOINT.is_dir(), f'Selected trained checkpoint missing: {CHECKPOINT}'
assert CHECKPOINT.resolve().is_relative_to(Path('/content/drive').resolve()), 'Use verified durable checkpoint storage'
print('Durable experiment:',ROOT);print('Selected checkpoint:',CHECKPOINT)

TASK06_ZIP=Path('/content/drive/MyDrive/NASN/task06/nasn-prototype-v1-20261012T111339Z-4e08cdbf/nasn-prototype-v1-20261012T111339Z-4e08cdbf-reports.zip')
assert TASK06_ZIP.is_file(), 'Provide the original completed Task 06 report ZIP path on Drive; never replace it with invented results'


## Exact managed environment

The Colab kernel can use another Python, but all research commands run in pinned
Python 3.12.14. CUDA PyTorch 2.8.0+cu126 must pass a real kernel check. Matplotlib
uses Agg explicitly to avoid inherited Colab inline-backend errors. Global Colab
packages remain separate. No new model download occurs.


In [ ]:
BOOTSTRAP=Path('/content/nasn-task07-bootstrap')
call([sys.executable,'-m','pip','install','--target',BOOTSTRAP,'uv==0.13.0'])
UV=BOOTSTRAP/'bin/uv'
BOOT_ENV={**os.environ,'UV_PYTHON_INSTALL_DIR':'/content/nasn-task07-python','UV_CACHE_DIR':'/content/nasn-task07-uv-cache','UV_PYTHON_BIN_DIR':'/content/nasn-task07-python-bin'}
call([UV,'python','install','--system-certs','3.12.14'],env=BOOT_ENV)
MANAGED=call([UV,'python','find','--managed-python','3.12.14'],env=BOOT_ENV).stdout.strip()
VENV=Path('/content/nasn-task07-venv');assert not VENV.exists(), 'Use a fresh runtime for environment setup'
call([MANAGED,'-m','venv',VENV]);PY=VENV/'bin/python'
call([PY,'-m','pip','install','torch==2.8.0+cu126','--index-url','https://download.pytorch.org/whl/cu126'])
call([PY,'-m','pip','install','-r',REPO/'requirements-task04.txt'])
ENV={**os.environ,'PYTHONPATH':str(REPO),'MPLBACKEND':'Agg','MPLCONFIGDIR':'/content/nasn-task07-matplotlib','XDG_CACHE_HOME':'/content/nasn-task07-cache','HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','CUBLAS_WORKSPACE_CONFIG':':4096:8'}
check=call([PY,'-m','pip','check'],cwd=REPO,env=ENV)
(ROOT/'dependency_check.txt').write_text(check.stdout+check.stderr)
(ROOT/'dependency_freeze.txt').write_text(call([PY,'-m','pip','freeze'],cwd=REPO,env=ENV).stdout)


In [ ]:
GPU_SCRIPT="""import json,platform,torch
assert platform.python_version()=='3.12.14'
assert torch.cuda.is_available() and torch.version.cuda, 'CUDA GPU unavailable'
free,total=torch.cuda.mem_get_info()
assert free>=2*1024**3, 'Less than 2 GiB free VRAM before model loading'
x=torch.ones((16,16),device='cuda');assert (x@x)[0,0].item()==16;torch.cuda.synchronize()
print(json.dumps({'gpu':torch.cuda.get_device_name(0),'free_vram_bytes':free,'total_vram_bytes':total,'cuda_build':torch.version.cuda,'torch':torch.__version__,'kernel_check_passed':True}))
"""
GPU=json.loads(call([PY,'-c',GPU_SCRIPT],cwd=REPO,env=ENV).stdout)
(ROOT/'gpu_runtime.json').write_text(json.dumps(GPU,indent=2)+'\n')
# Inference only: choose batch from free memory, not from performance metrics.
BATCH_SIZE=8 if GPU['free_vram_bytes']>=6*1024**3 else 4 if GPU['free_vram_bytes']>=3*1024**3 else 1
(ROOT/'runtime_choices.json').write_text(json.dumps({'code_ref':CODE_REF,'seed':42,'max_length':128,'inference_batch_size':BATCH_SIZE,'precision':'float32','training':False,'selection':'fixed methods, no parameter search'},indent=2)+'\n')
call([PY,'-m','pytest','-q'],cwd=REPO,env=ENV)


## Pinned training text and original Task 04 manifest

Only the official **training CSV** is restored. The existing Task 02 validation
manifest chooses its original rows; no split-generation command is used. Task 04
text restoration is explicit and verifies every existing registered ID, source/
target text hash and metadata field. It does not generate human decisions.
If the host or checksum fails, stop and retain the printed error. You can supply
the original pinned training file manually; never substitute another revision.


In [ ]:
FETCH_SCRIPT="""from pathlib import Path
from urllib.request import urlopen
from urllib.parse import urlsplit
from arsarcasm_audit.audit import COMMIT,HASHES,digest,sha256
p=Path('data/raw/training_data.csv');p.parent.mkdir(parents=True,exist_ok=True)
url=f'https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/{COMMIT}/ArSarcasm-v2/training_data.csv'
if not p.exists():
 try:
  with urlopen(url,timeout=60) as response: content=response.read()
 except Exception as exc:
  raise RuntimeError(f'Dataset host {urlsplit(url).hostname}: {type(exc).__name__}: {exc}') from exc
 assert digest(content)==HASHES['training_data.csv'], 'Pinned training download checksum differs'
 p.write_bytes(content)
assert sha256(p)==HASHES['training_data.csv'], 'Existing pinned training checksum differs'
print('Verified official training bytes only; test CSV not downloaded or opened')
"""
call([PY,'-c',FETCH_SCRIPT],cwd=REPO,env=ENV)
call([PY,'-m','nasn.task06','restore-task04','--output',ROOT/'task04-restored'],cwd=REPO,env=ENV)
# Extract ONLY exact selected tokenizer files to a separate verified directory.
TOKENIZER=ROOT/'selected-tokenizer';TOKENIZER.mkdir()
TOKENIZER_SCRIPT="""import shutil,sys,json
from pathlib import Path
from nasn.verified_tokenizer import FILES,verify
source,target,registry_path=map(Path,sys.argv[1:])
for name in FILES:shutil.copyfile(source/name,target/name)
verify(target,json.loads(registry_path.read_text()))
print('All selected tokenizer hashes verified')
"""
call([PY,'-c',TOKENIZER_SCRIPT,CHECKPOINT,TOKENIZER,BASELINE/'checkpoint_hashes.json'],cwd=REPO,env=ENV)
call([PY,'-m','nasn.task06','prepare','--preparation',ROOT/'task04-restored','--tokenizer-directory',TOKENIZER,'--output',ROOT/'v1-preparation'],cwd=REPO,env=ENV)

call([PY,'-m','nasn.task07_io','--archive',TASK06_ZIP,'--output',ROOT/'task06-verified'],cwd=REPO,env=ENV)
call([PY,'-m','nasn.task07','diagnose','--v1-prepared',ROOT/'v1-preparation','--tokenizer-directory',TOKENIZER,'--task06-intake',ROOT/'task06-verified','--output',ROOT/'v1-diagnosis'],cwd=REPO,env=ENV)
call([PY,'-m','tools.export_task07_examples','--v1-prepared',ROOT/'v1-preparation','--intake',ROOT/'task06-verified','--diagnosis',ROOT/'v1-diagnosis','--output',ROOT/'extra-examples'],cwd=REPO,env=ENV)
call([PY,'-m','nasn.task07','prepare','--v1-prepared',ROOT/'v1-preparation','--tokenizer-directory',TOKENIZER,'--output',ROOT/'preparation'],cwd=REPO,env=ENV)


## V2 inference and controlled scientific decision

Rules and the decision gate are frozen in `configs/nasn_v2.json` and
`docs/task07_protocol.md` before new v2 effects are inspected. Recognized negation
markers remain byte-identical. Run caps and context guards are heuristics; new
meaning-sensitive output is provisional. The 394 source-eligible cases have
user-declared reviews with unverified independence; 9,557 remain unreviewed and
10 source exclusions receive no robustness metrics. New review packages are blank.

The same model/tokenizer/config hashes and loading checks apply. RAW clean
predictions must reproduce Task 03B. Per-rule metrics and pooled original-ID
clustered paired intervals are computed without fitting or tuning. No automatic
winner becomes a verified semantic-robustness claim. On OOM preserve the failure,
reduce only inference batch size, and choose a fresh output directory.


In [ ]:
EVALUATION=ROOT/'evaluation'
call([PY,'-m','nasn.task07','run','--prepared',ROOT/'preparation','--checkpoint',CHECKPOINT,'--batch-size',BATCH_SIZE,'--output',EVALUATION],cwd=REPO,env=ENV)
status=json.loads((EVALUATION/'run_status.json').read_text())
assert status['status']=='completed' and status['metrics_computed'] and status['checkpoint_reload_verified']
assert status['training_performed'] is False and status['test_text_read'] is False and status['test_evaluated'] is False
print(json.dumps(status,indent=2))
print((EVALUATION/'scientific_decision.json').read_text())


## Durable reports and private review/examples

The report ZIP contains numeric model outputs, metrics, hashes, decision status,
runtime and text-free diagnostic tables. Return this ZIP for independent intake.
Do not publish weights or raw tweet/reviewer text to GitHub. A separate **private**
ZIP contains all v1 guard-difference examples and new blank human review packages;
retain it privately on Drive and send ONLY `new_human_reviewer_a.zip` and
`new_human_reviewer_b.zip` to the respective independent
reviewers. Review semantics/negation/emphasis rather than trusting normalization.
No annotation is automatically marked human-confirmed. Downloading reports does
not complete human validation or establish reviewer independence.


In [ ]:
VERIFY_SCRIPT="""import sys
from nasn.task05 import registry
for path in sys.argv[1:]:registry(path)
print('Durable Drive registries verified')
"""
call([PY,'-c',VERIFY_SCRIPT,ROOT/'preparation',ROOT/'preparation/review-package',EVALUATION],cwd=REPO,env=ENV)
import zipfile
REPORT_ZIP=ROOT/(RUN_ID+'-reports.zip')
names=['evaluation/'+p.name for p in EVALUATION.iterdir() if p.is_file()]
names+=['preparation/'+name for name in ['evidence.json','run_status.json','normalization_manifest.csv.gz','normalization_rules.json','character_token_audit.csv']]
names+=['v1-diagnosis/'+name for name in ['pipeline_equality.csv','all_B3_B4_differences.csv','all_elongation_token_exceptions.csv','guard_difference_summary.csv','annotation_quality_status.json','evidence.json']]
names+=['gpu_runtime.json','runtime_choices.json','dependency_check.txt','dependency_freeze.txt']
checks={name:hashlib.sha256((ROOT/name).read_bytes()).hexdigest() for name in names}
with zipfile.ZipFile(REPORT_ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for name in names:z.write(ROOT/name,arcname=name)
    z.writestr('report_checksums.json',json.dumps(checks,indent=2)+'\n')
with zipfile.ZipFile(REPORT_ZIP) as z:
    for name,digest in checks.items():assert hashlib.sha256(z.read(name)).hexdigest()==digest
PRIVATE_ZIP=ROOT/(RUN_ID+'-PRIVATE-examples-review.zip')
with zipfile.ZipFile(PRIVATE_ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ROOT/'v1-diagnosis/all_B3_B4_differences_private.csv',arcname='all_B3_B4_differences_private.csv')
    z.write(ROOT/'extra-examples/all_elongation_exceptions_private.csv',arcname='all_elongation_exceptions_private.csv')
    z.write(ROOT/'preparation/new_human_review_queue.csv',arcname='new_human_review_queue.csv')
    for reviewer in ['a','b']:
        z.write(ROOT/f'preparation/new_human_reviewer_{reviewer}.zip',arcname=f'new_human_reviewer_{reviewer}.zip')
    for p in (ROOT/'preparation/review-package').iterdir():
        if p.is_file():z.write(p,arcname='review-package/'+p.name)
print('Durable lightweight report:',REPORT_ZIP)
print('Private examples/review archive:',PRIVATE_ZIP)
from google.colab import files
files.download(str(REPORT_ZIP))
